# NB03 – Daten Analyse

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Senthuran Elankeswaran | **Datum:** März 2026

---

*Preisstruktur, [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Simulation, Wirtschaftlichkeit, [Netzentlastung](../organisation/O_02_Glossar.ipynb#g-netzentlastung), Spread-Zeitreihe.*


| [← NB02 Daten Bereinigung](02_Daten_Bereinigung.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [NB04 Visualisierungen →](04_Visualisierungen.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_NB_03'></a>

1. [Initialisierung](#initialisierung_NB_03)
2. [Preisstruktur-Analyse](#preisstruktur-analyse_NB_03)
3. [Batterie-Dispatch-Simulation](#batterie-dispatch-simulation_NB_03)
4. [Wirtschaftlichkeitsrechnung](#wirtschaftlichkeitsrechnung_NB_03)
5. [Netzentlastungsszenarien](#netzentlastungsszenarien_NB_03)
6. [Spread- & Volatilitäts-Zeitreihe](#spread-volatilitaets-zeitreihe_NB_03)
7. [Abschluss & Verifikation](#abschluss-verifikation_NB_03)


## Initialisierung<a id='initialisierung_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Lädt `../sync/config.json` (SSOT), setzt Verzeichnispfade, Simulation- und Wirtschaftlichkeitsparameter sowie Datenzeitraum und Simulationsergebnisse aus `../sync/transfer.json`.

**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json`, setzt Pfade
und liest `FORCE_RELOAD`, `LIFETIME` und `ZIEL_ROI` als lokale Aliases.


In [ ]:
import os, sys, warnings, json
import numpy  as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

# Versionen anzeigen für Reproduzierbarkeit
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"\U0001f4c5 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")

In [ ]:
# ── ../sync/config.json laden (Single Source of Truth) ───────────────────────────────
# NEVER hardcode MODE oder FORCE_RELOAD hier — immer aus ../sync/config.json lesen
with open('../sync/config.json') as _f:
    CFG = json.load(_f)

MODE         = CFG['mode']
FORCE_RELOAD = CFG['force_reload']

# ── Sim-Parameter aus ../sync/config.json ────────────────────────────────────────────
_sim         = CFG['pflicht']['simulation']
CHARGE_Q     = _sim['charge_quantile']      # 0.25
DISCHARGE_Q  = _sim['discharge_quantile']   # 0.75
SOC_MIN_PCT  = _sim['soc_min_pct']          # 0.05
SOC_MAX_PCT  = _sim['soc_max_pct']          # 0.95
EFFICIENCY   = _sim['efficiency_roundtrip'] # 0.92

# ── Wirtschaftlichkeits-Parameter ─────────────────────────────────────────────
_w           = CFG['pflicht']['wirtschaftlichkeit']
CAPEX_EUR_KWH = _w['capex_eur_kwh']
OPEX_RATE    = _w['opex_rate']
LIFETIME     = _w['lifetime_j']
ZIEL_ROI_PCT = round(100 / LIFETIME, 2)  # Ziel-ROI = 1/lifetime_j: ROI der nötig ist, um BE in LIFETIME Jahren zu erreichen

# ── Szenario-Parameter (Gleichzeitigkeit) ─────────────────────────────────────
_sz           = CFG['szenarien']
GLEICHZEITIGKEIT = _sz['gleichzeitigkeit_aktiv']          # 'pessimistisch'|'realistisch'|'optimistisch'
RATE          = _sz['optionen'][GLEICHZEITIGKEIT]['rate']
SZ_OPT        = _sz['optionen'][GLEICHZEITIGKEIT]         # alle Parameter des aktiven Szenarios
CH_SPITZENLAST_GW = _sz['ch_spitzenlast_gw']

DIR_RAW          = os.path.join('../data', 'raw')
DIR_PROCESSED    = os.path.join('../data', 'processed')
# intermediate: Szenario-abhängige Dateien in Unterordner
DIR_INTER_BASE   = os.path.join('../data', 'intermediate')
DIR_INTER_SZ     = os.path.join(DIR_INTER_BASE, GLEICHZEITIGKEIT)   # z.B. data/intermediate/realistisch/
DIR_INTER        = DIR_INTER_BASE  # szenario-unabhängige Dateien (spread, import/export)
DATAINDEX        = '../sync/dataindex.csv'

for d in [DIR_RAW, DIR_PROCESSED, DIR_INTER_BASE, DIR_INTER_SZ]:
    os.makedirs(d, exist_ok=True)

print(f'../sync/config.json  : geladen')
print(f'MODE         : {MODE}')
print(f'Gleichz.     : {GLEICHZEITIGKEIT} ({RATE*100:.0f}%)')
print(f'raw          : {os.path.abspath(DIR_RAW)}')
print(f'processed    : {os.path.abspath(DIR_PROCESSED)}')
print(f'intermediate : {os.path.abspath(DIR_INTER_SZ)} (szenario-abhängig)')
# -- Transfer: Ergebnisse aus ../sync/transfer.json laden ----------------------------
_tf_path = '../sync/transfer.json'
if os.path.exists(_tf_path) and os.path.getsize(_tf_path) > 0:
    TF      = json.load(open(_tf_path))
    _dt     = TF.get('datenzeitraum', {})
    _sim    = TF.get('simulation', {})
    TF_N_YEARS  = _dt.get('n_years', None)
    TF_START    = _dt.get('start_date', 'unbekannt')
    TF_END      = _dt.get('end_date', 'unbekannt')
    TF_SPREAD   = _sim.get('spread_mean_eur_mwh', None)
    TF_ECON     = _sim.get('wirtschaftlichkeit', {})
    TF_HYB      = TF.get('hybrid_simulation', {}).get('ergebnisse', {})
    TF_KUER     = CFG.get('kuer_aktiv', {})
    print(f"../sync/transfer.json: {TF_START} – {TF_END} ({TF_N_YEARS} Jahre) | Spread: {TF_SPREAD} EUR/MWh")
else:
    TF = {}; TF_N_YEARS = None; TF_START = TF_END = 'unbekannt'
    TF_SPREAD = None; TF_ECON = {}; TF_HYB = {}; TF_KUER = {}
    print('⚠️  ../sync/transfer.json nicht gefunden — NB01/NB02 zuerst ausführen')

**Transfer NB01 → NB02:** Datenzeitraum (`n_years`) aus `../sync/transfer.json` lesen —
SSOT für alle Jahresdurchschnitts-Berechnungen. Fehlt die Datei: Fallback auf Selbstberechnung.


In [ ]:
# -- Transfer: n_years von NB01 übernehmen (SSOT: gleiche Methode überall) ----
_tf_path = '../sync/transfer.json'
if os.path.exists(_tf_path) and os.path.getsize(_tf_path) > 0:
    _tf_r = json.load(open(_tf_path))
    n_years = _tf_r.get('datenzeitraum', {}).get('n_years', None)
    _sz_tf  = _tf_r.get('datenzeitraum', {}).get('_szenario_aktiv', None)
    if n_years:
        print(f'n_years aus ../sync/transfer.json (NB01): {n_years:.3f}')
    if _sz_tf and _sz_tf != SZ_AKTIV:
        print(f'⚠️  Szenario-Konflikt: transfer={_sz_tf} vs config={SZ_AKTIV}')
        print('   NB01 mit aktuellem Szenario neu ausführen!')
else:
    n_years = None
    print('⚠️  ../sync/transfer.json nicht gefunden — NB01 zuerst ausführen')


**Hilfsfunktionen:** `log_dataindex()` für das Datenprotokoll (identisch zu NB01,
hier neu definiert damit NB02 eigenständig ausführbar bleibt).


In [ ]:
# ── ../sync/dataindex.csv Helper ───────────────────────────────────────────────────────
def log_dataindex(filename, source_url, local_path, data_type,
                  rows=None, size_kb=None, status='active', note=''):
    """
    Zentrales Datenquellen-Register. Jeder Eintrag bleibt als Historie erhalten.
    status: active | superseded | error | partial
    """
    ts = datetime.utcnow().isoformat(timespec='seconds') + 'Z'
    if os.path.exists(DATAINDEX):
        df_idx = pd.read_csv(DATAINDEX)
        mask   = (df_idx['filename'] == filename) & (df_idx['status'] == 'active')
        if mask.any():
            df_idx.loc[mask, 'status']        = 'superseded'
            df_idx.loc[mask, 'superseded_at'] = ts
    else:
        df_idx = pd.DataFrame(columns=['timestamp','filename','source_url','local_path',
                                        'data_type','rows','size_kb','status','superseded_at','note'])
    row = {'timestamp': ts, 'filename': filename, 'source_url': source_url,
           'local_path': local_path, 'data_type': data_type, 'rows': rows,
           'size_kb': round(size_kb,1) if size_kb else None,
           'status': status, 'superseded_at': '', 'note': note}
    pd.concat([df_idx, pd.DataFrame([row])], ignore_index=True).to_csv(DATAINDEX, index=False)
    print(f'  dataindex: {filename} [{status}]')

def log_missing(source, reason):
    ts = datetime.utcnow().isoformat(timespec='seconds')
    with open(os.path.join('../data','missing.txt'), 'a') as f:
        f.write(f'[{ts}] MISSING {source}: {reason}\n')
    log_dataindex(os.path.basename(source), source, '', 'raw', status='error', note=reason)
    print(f'  missing.txt: {reason}')

print('dataindex Helper bereit.')

def needs_rebuild(filepath, min_rows, ds_key):
    """True wenn Datei fehlt, zu wenig Zeilen, oder FORCE_RELOAD gesetzt."""
    if FORCE_RELOAD.get(ds_key, False):
        print(f'  FORCE_RELOAD={ds_key} → neu erzeugen')
        return True
    if not os.path.exists(filepath):
        return True
    try:
        n = sum(1 for _ in open(filepath)) - 1
        if n < min_rows:
            print(f'  Zu wenig Zeilen ({n} < {min_rows}) → neu erzeugen')
            return True
    except: return True
    return False

print('Helpers bereit.')


**Rohdaten laden:** Spot-Preise und Netzlast aus `data/processed/` (bereinigt, NB02-Output) und Netzlast aus `data/raw/` einlesen;
Zeitfeatures (`hour`, `month`, `season`) berechnen.


In [ ]:
# ── Daten laden (NB02-Output) ────────────────────────────────────────────────
# Spot-Preise: aus data/processed/ (bereinigt von NB02), nicht raw!
# Netzlast:    aus data/raw/ (keine Bereinigung nötig — NB01-Output)
df_prices = pd.read_csv(os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv'))
df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)

# Zeitfeatures (werden HIER berechnet — NB02 speichert nur bereinigte Preise)
if 'hour' not in df_prices.columns:
    df_prices['hour']  = df_prices['timestamp'].dt.hour
if 'month' not in df_prices.columns:
    df_prices['month'] = df_prices['timestamp'].dt.month
if 'season' not in df_prices.columns:
    _season_map = {12:'Winter', 1:'Winter',  2:'Winter',
                    3:'Frühling',4:'Frühling',5:'Frühling',
                    6:'Sommer',  7:'Sommer',  8:'Sommer',
                    9:'Herbst', 10:'Herbst', 11:'Herbst'}
    df_prices['season'] = df_prices['month'].map(_season_map)

df_load = pd.read_csv(os.path.join(DIR_RAW, 'ch_netzlast_raw.csv'))
df_load['timestamp'] = pd.to_datetime(df_load['timestamp'], utc=True)

print(f'MODE       : {MODE}')
print(f'Preisdaten : {len(df_prices):,} Zeilen | {list(df_prices.columns)}')
print(f'Lastdaten  : {len(df_load):,} Zeilen  | {list(df_load.columns)}')


---
## 2. Preisstruktur-Analyse <a id='preisstruktur-analyse_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Berechnet stündliche Durchschnittspreise und den Intra-Tag-Spread (p75 − p25).
Der Spread ist der direkte Erlöstreiber — je höher, desto rentabler die Arbitrage.


In [ ]:
# ── Tagesprofil & Spread ──────────────────────────────────────────────────────
h_stats   = df_prices.groupby('hour')['price_eur_mwh'].agg(mean='mean',std='std').round(2)
low_h     = h_stats['mean'].nsmallest(4).index.tolist()
hi_h      = h_stats['mean'].nlargest(4).index.tolist()
spread    = h_stats.loc[hi_h,'mean'].mean() - h_stats.loc[low_h,'mean'].mean()

print(f'Guenstigste Stunden  : {sorted(low_h)}  Ø {h_stats.loc[low_h,"mean"].mean():.1f} EUR/MWh')
print(f'Teuerste Stunden     : {sorted(hi_h)}   Ø {h_stats.loc[hi_h,"mean"].mean():.1f} EUR/MWh')
print(f'Arbitrage-Spread (Ø) : {spread:.1f} EUR/MWh')
sn = {0:'Winter',1:'Frühling',2:'Sommer',3:'Herbst'}
for s, avg in df_prices.groupby('season')['price_eur_mwh'].mean().items():
    print(f'  {sn[s]:<10}: {avg:.1f} EUR/MWh')


---
## 3. Batterie-Dispatch-Simulation <a id='batterie-dispatch-simulation_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Tagesbasierte Schwellenwert-Strategie: Laden bei Preis ≤ [p25](../organisation/O_02_Glossar.ipynb#g-p25p75), Einspeisen bei Preis ≥ [p75](../organisation/O_02_Glossar.ipynb#g-p25p75).
Zuerst die [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Funktion definieren, dann alle vier Segmente über den
gesamten Datensatz simulieren und den Jahresdurchschnitt bilden.


In [ ]:
# ── Dispatch-Funktion ─────────────────────────────────────────────────────────
# Performance-Optimierungen:
#   1. Tages-Quantile werden EINMAL vorab berechnet (nicht pro Zeile) → O(n) statt O(n²)
#   2. NumPy-Arrays statt iterrows() → ~50x schneller
#   3. tqdm Progress-Bar pro Segment

def simulate_battery(prices_df, capacity_kwh, power_kw,
                     efficiency, charge_q, discharge_q,
                     soc_min_pct, soc_max_pct):
    """
    Schwellenwertmodell (tagesbasiert):
      LADEN      : Preis < p(charge_q) des Tages  UND  SoC < soc_max_pct
      EINSPEISEN : Preis > p(discharge_q) des Tages  UND  SoC > soc_min_pct
    Alle Parameter kommen aus ../sync/config.json (SSOT) — keine Defaults im Code.

    Break-even-Bedingung (Dispatch lohnt sich nur wenn):
        p(discharge_q) × η  >  p(charge_q)
    Äquivalent:
        Spread  >  Spread_min  =  p(charge_q) × (1/η − 1)
    Beispiel (η=0.92, p25=25 EUR/MWh): Spread_min ≈ 2.2 EUR/MWh.
    Tage, an denen kein Preis die Schwelle unterschreitet (charge) oder überschreitet
    (discharge), bleiben vollständig idle — kein Laden, kein Entladen.
    → Siehe Glossar: O_02_Glossar.ipynb#g-dispatch-breakeven

    Optimierung: Tages-Quantile werden einmal vorab berechnet,
    nicht für jede Zeile neu — reduziert Laufzeit von O(n²) auf O(n).
    """
    import numpy as np

    # ── Schritt 1: Tages-Quantile vorab berechnen ─────────────────────────────
    df = prices_df[['timestamp', 'price_eur_mwh']].copy()
    df['date'] = df['timestamp'].dt.date
    day_q = df.groupby('date')['price_eur_mwh'].agg(
        p_lo=lambda x: x.quantile(charge_q),
        p_hi=lambda x: x.quantile(discharge_q),
    )
    df = df.join(day_q, on='date')

    # ── Schritt 2: NumPy-Arrays — kein iterrows() ─────────────────────────────
    prices   = df['price_eur_mwh'].to_numpy()
    p_los    = df['p_lo'].to_numpy()
    p_his    = df['p_hi'].to_numpy()
    n        = len(prices)

    soc_max  = capacity_kwh * soc_max_pct  # aus ../sync/config.json (SSOT)
    soc_min  = capacity_kwh * soc_min_pct  # aus ../sync/config.json (SSOT)
    sqrt_eff = efficiency ** 0.5
    soc      = capacity_kwh * 0.5   # Startzustand

    actions    = np.empty(n, dtype='U10')
    cashflows  = np.zeros(n)
    grid_delta = np.zeros(n)

    # ── Schritt 3: Simulation ────────────────────────────────────────────────
    for idx in range(n):
        price = prices[idx]
        if price <= p_los[idx] and soc < soc_max:
            e = min(power_kw, (soc_max - soc) / sqrt_eff)
            soc += e * sqrt_eff
            actions[idx]    = 'charge'
            cashflows[idx]  = -(e * price / 1000)
            grid_delta[idx] = +e
        elif price >= p_his[idx] and soc > soc_min:
            e = min(power_kw, soc * sqrt_eff)
            soc -= e / sqrt_eff
            actions[idx]    = 'discharge'
            cashflows[idx]  = +(e * sqrt_eff * price / 1000)
            grid_delta[idx] = -e
        else:
            actions[idx] = 'idle'

    return pd.DataFrame({
        'timestamp':    df['timestamp'].values,
        'action':       actions,
        'cashflow_eur': cashflows,
        'grid_delta_kw': grid_delta,
    })

print('Dispatch-Funktion bereit (optimiert: O(n), numpy-Loop).')


**Dispatch-Simulation:** Alle vier Segmente (Privat / Gewerbe / Industrie / Utility)
über den gesamten Datensatz simulieren; Jahresdurchschnitt aus `n_years` bilden.


In [ ]:
import json  # Re-Import mit lokalem Alias (Simulations-Zelle)

# ── Simulation aller Segmente ─────────────────────────────────────────────────
# Simuliert alle verfügbaren Daten und bildet den Jahresdurchschnitt (sum / n_years).
# Optimierte Dispatch-Funktion: O(n) statt O(n²), tqdm Progress-Bar pro Segment.
# Typische Laufzeit: ~5-15 Sekunden für alle 4 Segmente.

SEGMENTS = [('Privat_10kWh',10,5,200_000),('Gewerbe_100kWh',100,30,10_000),
            ('Industrie_1MWh',1_000,200,1_000),('Utility_10MWh',10_000,1_000,100)]

# Alle verfügbaren Preisdaten (kein df_sample mehr)
df_sim = df_prices.copy()
# n_years aus ../sync/transfer.json (NB01 SSOT); Fallback: selbst berechnen
if n_years is None:
    n_years = df_sim['timestamp'].dt.year.nunique()
    print(f'n_years Fallback (selbst berechnet): {n_years}')
# n_years jetzt verfügbar:
results = {}

print(f'Simulationszeitraum: {df_sim["timestamp"].min().date()} – '
      f'{df_sim["timestamp"].max().date()} ({n_years} Jahre)')
print(f'{"Segment":<22} {"Jahreserloes":>13} {"EUR/kWh/J":>11} {"Lade-h/J":>10} {"Einsp-h/J":>10}')
print('-'*70)

for name, cap, pwr, units in SEGMENTS:
    res         = simulate_battery(
                      df_sim, cap, pwr,
                      efficiency=EFFICIENCY,
                      charge_q=CHARGE_Q,
                      discharge_q=DISCHARGE_Q,
                      soc_min_pct=SOC_MIN_PCT,
                      soc_max_pct=SOC_MAX_PCT)
    annual_rev  = res['cashflow_eur'].sum() / n_years   # Ø über alle Jahre
    rev_per_kwh = annual_rev / cap
    n_ch = (res['action']=='charge').sum()    / n_years
    n_di = (res['action']=='discharge').sum() / n_years
    results[name] = dict(cap=cap, pwr=pwr, units=units, annual_rev=annual_rev,
                          rev_per_kwh=rev_per_kwh, n_charge=n_ch, n_discharge=n_di)
    print(f'{name:<22} {annual_rev:>12.0f}€ {rev_per_kwh:>10.2f} '
          f'{n_ch:>10.0f} {n_di:>10.0f}')

print('\nSimulation abgeschlossen (Ø Jahreserlös über gesamten Datenzeitraum).')


---
## 4. Wirtschaftlichkeitsrechnung <a id='wirtschaftlichkeitsrechnung_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Berechnet [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex), jährliche Erlöse, [ROI](../organisation/O_02_Glossar.ipynb#g-roi) und Amortisationsdauer für alle Segmente.
Ergebnis: `wirtschaftlichkeit.csv` und `netzentlastung_szenarien.csv` in `intermediate/<szenario>/`.


In [ ]:
# ── CAPEX / ROI / Amortisation → intermediate/ ────────────────────────────────
# CAPEX_EUR_KWH, OPEX_RATE, LIFETIME bereits aus ../sync/config.json geladen (Setup-Zelle)

econ = {}
print(f'{"Segment":<22} {"CAPEX":>9} {"Jährl.Erlös":>12} {"OPEX":>8} {"Netto":>8} {"Amort.":>9} {"ROI":>7}')
print('-'*83)

for name, res in results.items():
    cap, annual_rev = res['cap'], res['annual_rev']
    capex = CAPEX_EUR_KWH[name]*cap
    opex  = capex*OPEX_RATE
    net   = annual_rev-opex
    pb    = capex/net if net>0 else float('inf')
    roi   = net/capex*100
    econ[name] = dict(segment=name, capex=capex, annual_rev=annual_rev,
                       opex_annual=opex, net_annual=net,
                       payback_years=pb, roi_pct=roi, rev_per_kwh=res['rev_per_kwh'])
    pb_s = f'{pb:.1f}J' if pb<50 else '>50J'
    print(f'{name:<22} {capex:>8.0f}€ {annual_rev:>11.0f}€ {opex:>7.0f}€ '
          f'{net:>7.0f}€ {pb_s:>9} {roi:>6.1f}%')

df_econ = pd.DataFrame(list(econ.values()))
ECON_FILE = os.path.join(DIR_INTER_SZ, 'wirtschaftlichkeit.csv')  # szenario-abhängig
df_econ.to_csv(ECON_FILE, index=False)
log_dataindex('wirtschaftlichkeit.csv','NB2:dispatch', ECON_FILE,'intermediate',
              rows=len(df_econ), size_kb=os.path.getsize(ECON_FILE)/1024, note=f'MODE={MODE}')
print(f'\nGespeichert: {ECON_FILE}')
print('Hinweis: Nur Arbitrage-Erloese – Regelenergie (FCR/aFRR) nicht eingerechnet.')


**Verifikation:** [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex), [ROI](../organisation/O_02_Glossar.ipynb#g-roi) und Amortisation der vier Segmente tabellarisch prüfen.


In [ ]:
# ── Verifikation: Wirtschaftlichkeit ────────────────────────────────────────
print(f'Segmente: {len(df_econ)}')
df_econ[['segment','capex','net_annual','roi_pct','payback_years']].head()


---
## 5. Netzentlastungsszenarien <a id='netzentlastungsszenarien_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

### Modell: Aggregierter Rollout-Effekt

Die vier Szenarien beschreiben mögliche Rollout-Zustände des CH-Batteriemarkts.
Die Zahlen sind **Schätzungen** basierend auf Markttrends und [BFE](../organisation/O_02_Glossar.ipynb#g-bfe)-Prognosen —
keine offiziellen Zielvorgaben.

| Szenario | Jahr | Privat | Gewerbe | Industrie |
|---|---|---|---|---|
| Status Quo | 2024 | 0 | 0 | 0 |
| Moderat | 2027 | 50'000 | 2'000 | 200 |
| Ambitioniert | 2030 | 200'000 | 8'000 | 800 |
| Transformativ | 2035 | 800'000 | 30'000 | 2'000 |

**Berechnung [Netzentlastung](../organisation/O_02_Glossar.ipynb#g-netzentlastung):**
```
Entlastung [MW] = (n_privat×5kW + n_gewerbe×30kW + n_industrie×200kW) / 1000 × Rate
```

**Gleichzeitigkeits-Schalter** (`GLEICHZEITIGKEIT` in der Code-Zelle):

| Modus | Rate | Annahme |
|---|---|---|
| `'optimistisch'` | 70%⚙ | Koordinierter [VPP](../organisation/O_02_Glossar.ipynb#g-vpp)-[Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch) — alle Batterien reagieren aufs gleiche Signal |
| `'realistisch'` | 40%⚙ | Unkoordinierter Markt — jede Batterie dispatcht eigenständig |

> **Default:** `'realistisch'` — die Berichte (NB4) referenzieren den gesetzten Wert
> automatisch. Um die optimistische Variante zu berechnen: Schalter umstellen und
> NB2 + NB3 neu ausführen.


In [ ]:
# ── Szenarien → intermediate/<szenario>/ ──────────────────────────────────────
# GLEICHZEITIGKEIT und RATE kommen aus ../sync/config.json (bereits in Setup geladen)
# Szenario-Outputs in DIR_INTER_SZ = data/intermediate/<gleichzeitigkeit>/

KW = {'Privat_10kWh': 5, 'Gewerbe_100kWh': 30, 'Industrie_1MWh': 200}

SCENARIOS = [
    ('Status Quo (2024)',    0,
     SZ_OPT.get('n_privat_2027',0)*0,    # 0 für Status Quo
     SZ_OPT.get('n_gewerbe_2027',0)*0,
     SZ_OPT.get('n_industrie_2027',0)*0),
    ('Moderat (2027)',
     SZ_OPT.get('n_privat_2027', 50_000),
     SZ_OPT.get('n_gewerbe_2027', 2_000),
     SZ_OPT.get('n_industrie_2027', 200)),
    ('Ambitioniert (2030)',
     SZ_OPT.get('n_privat_2030', 200_000),
     SZ_OPT.get('n_gewerbe_2030', 8_000),
     SZ_OPT.get('n_industrie_2030', 800)),
    ('Transformativ (2035)',
     SZ_OPT.get('n_privat_2035', 800_000),
     SZ_OPT.get('n_gewerbe_2035', 30_000),
     SZ_OPT.get('n_industrie_2035', 2_000)),
]

print(f'Gleichzeitigkeit : {GLEICHZEITIGKEIT} ({RATE*100:.0f}%)')
print(f'CH Systemspitze  : {CH_SPITZENLAST_GW} GW')
print(f'Szenario-Output  : {DIR_INTER_SZ}')
print()
print(f'{"Szenario":<28} {"Entlastung [MW]":>16} {"Reduktion":>10}')
print('-'*58)

rows = []
for sc_data in SCENARIOS:
    if len(sc_data) == 4:
        sc, n_p, n_g, n_i = sc_data
    else:
        sc = sc_data[0]; n_p = sc_data[1]; n_g = sc_data[2]; n_i = sc_data[3]
    mw = (n_p*KW['Privat_10kWh'] + n_g*KW['Gewerbe_100kWh'] +
          n_i*KW['Industrie_1MWh']) / 1000 * RATE
    rows.append({
        'szenario':            sc,
        'gleichzeitigkeit':    GLEICHZEITIGKEIT,
        'rate_pct':            RATE * 100,
        'n_privat':            n_p,
        'n_gewerbe':           n_g,
        'n_industrie':         n_i,
        'entlastung_mw':       round(mw, 1),
        'neue_spitzenlast_gw': round(max(0, CH_SPITZENLAST_GW - mw/1000), 3),
        'reduktion_pct':       round(mw / 1000 / CH_SPITZENLAST_GW * 100, 2),
    })
    print(f'{sc:<28} {mw:>15.0f} MW  {mw/1000/CH_SPITZENLAST_GW*100:>9.1f}%')

df_sz = pd.DataFrame(rows)
SZ_FILE = os.path.join(DIR_INTER_SZ, 'netzentlastung_szenarien.csv')
df_sz.to_csv(SZ_FILE, index=False)
log_dataindex('netzentlastung_szenarien.csv', 'NB02:szenario', SZ_FILE,
              'intermediate',  # Szenario steht in note= (data_type bleibt generisch)
              rows=len(df_sz),
              size_kb=os.path.getsize(SZ_FILE)/1024,
              note=f'MODE={MODE}, Gleichzeitigkeit={GLEICHZEITIGKEIT} ({RATE*100:.0f}%)')
print(f'\nGespeichert: {SZ_FILE}')


---
## 6. Spread- & Volatilitäts-Zeitreihe <a id='spread-volatilitaets-zeitreihe_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Berechnet monatlich: Intra-Tag-Spread ([p75−p25](../organisation/O_02_Glossar.ipynb#g-p25p75) Median), [Volatilität](../organisation/O_02_Glossar.ipynb#g-volatilitaet) und
Negativpreis-Stunden → `spread_zeitreihe.csv` in `intermediate/`.

| Kennzahl | Berechnung | Bedeutung |
|---|---|---|
| `spread_median` | Median(p75−p25 pro Tag) | Typisches tägliches Arbitrage-Fenster |
| `volatility` | Std. Abweichung Stundenpreise | Allgemeine Marktvolatilität |
| `neg_price_h` | Stunden mit Preis < 0 | Solar-Überschuss-Indikator |

> Chart 7 (NB3) visualisiert diese Zeitreihe mit Trendlinie.
> Jährlich neu ausführen um Trend zu aktualisieren.


In [ ]:
# ── 6. Spread/Volatilität Zeitreihe → intermediate/ ─────────────────────────
SPREAD_FILE = os.path.join(DIR_INTER, 'spread_zeitreihe.csv')

if not needs_rebuild(SPREAD_FILE, 12, 'spread_ts'):
    print(f'spread_zeitreihe.csv vorhanden ({os.path.getsize(SPREAD_FILE)/1024:.0f} KB) – kein Rebuild.')
    df_spread = pd.read_csv(SPREAD_FILE, parse_dates=['yearmonth'])
else:
    print('Berechne Spread-Zeitreihe...')
    CLEAN_FILE = os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv')
    df_clean = pd.read_csv(CLEAN_FILE, parse_dates=['timestamp'])
    df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], utc=True)
    df_clean['date']      = df_clean['timestamp'].dt.date
    df_clean['yearmonth'] = df_clean['timestamp'].dt.to_period('M').dt.to_timestamp()

    # Intra-Tag Spread pro Tag
    day_stats = df_clean.groupby('date')['price_eur_mwh'].agg(
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
    ).reset_index()
    day_stats['spread']    = day_stats['p75'] - day_stats['p25']
    day_stats['yearmonth'] = pd.to_datetime(day_stats['date']).dt.to_period('M').dt.to_timestamp()

    # Monatliche Aggregation
    spread_m = day_stats.groupby('yearmonth')['spread'].median().reset_index()
    spread_m.columns = ['yearmonth', 'spread_median']

    vol_m = df_clean.groupby('yearmonth').agg(
        volatility  = ('price_eur_mwh', 'std'),
        mean_price  = ('price_eur_mwh', 'mean'),
        neg_price_h = ('price_eur_mwh', lambda x: (x < 0).sum()),
    ).reset_index()

    df_spread = spread_m.merge(vol_m, on='yearmonth')
    df_spread.to_csv(SPREAD_FILE, index=False)
    log_dataindex('spread_zeitreihe.csv', 'NB2:spread_analyse', SPREAD_FILE,
                  'intermediate', rows=len(df_spread),
                  size_kb=os.path.getsize(SPREAD_FILE)/1024,
                  note='Monatlicher Intra-Tag-Spread, Volatilität, Negativpreise')
    print(f'Gespeichert: {SPREAD_FILE} | {len(df_spread)} Monate')
    print(f'  Zeitraum: {df_spread["yearmonth"].min().strftime("%Y-%m")} – '
          f'{df_spread["yearmonth"].max().strftime("%Y-%m")}')
    print(f'  Ø Spread: {df_spread["spread_median"].mean():.1f} EUR/MWh '
          f'| Min: {df_spread["spread_median"].min():.1f} '
          f'| Max: {df_spread["spread_median"].max():.1f}')
    print(f'  Negativpreis-Stunden total: {df_spread["neg_price_h"].sum():,}')


**Verifikation:** Spread-Zeitreihe (monatlich) auf Shape und Wertebereich prüfen.


In [ ]:
# ── Verifikation: Spread-Zeitreihe ───────────────────────────────────────────
print(f'Shape  : {df_spread.shape}')
print(f'Spalten: {list(df_spread.columns)}')
df_spread.head(3)


**Transfer NB02 → downstream:** Simulation-Kennzahlen und Datenzeitraum in
`../sync/transfer.json` schreiben — wird von NB04, K_00, K_05, K_10, K_99 gelesen.


In [ ]:
# -- Transfer: Simulation-Ergebnisse in ../sync/transfer.json schreiben ---------------
import json  # Re-Import mit lokalem Alias (Transfer-Zelle)
_tf_path = '../sync/transfer.json'
_tf = json.loads(open(_tf_path).read() or '{}') if os.path.exists(_tf_path) and os.path.getsize(_tf_path) > 0 else {}
_tf.setdefault('simulation', {})

# Spread-Kennzahlen aus spread_zeitreihe.csv
if os.path.exists(SPREAD_FILE):
    _df_sp = pd.read_csv(SPREAD_FILE)
    _tf['simulation']['spread_mean_eur_mwh']       = round(float(_df_sp['spread_median'].mean()), 2)
    _tf['simulation']['spread_p25_mean']            = round(float(_df_sp['spread_median'].quantile(0.25)), 2)
    _tf['simulation']['spread_p75_mean']            = round(float(_df_sp['spread_median'].quantile(0.75)), 2)
    _tf['simulation']['neg_price_hours_per_year']   = round(float(_df_sp['neg_price_h'].mean()), 1)

# n_years aus Preisdaten
_tf['simulation']['n_years_sim'] = n_years

# Wirtschaftlichkeit je Segment
_tf['simulation']['wirtschaftlichkeit'] = {}
for _, row in df_econ.iterrows():
    _tf['simulation']['wirtschaftlichkeit'][row['segment']] = {
        'roi_pct':      round(float(row['roi_pct']), 2),
        'net_annual':   round(float(row['net_annual']), 1),
        'annual_rev':   round(float(row['annual_rev']), 1),
        'capex':        round(float(row['capex']), 0),
        'payback_years': round(float(row['payback_years']), 1) if row['payback_years'] < 999 else 999,
    }

with open(_tf_path, 'w') as _f: json.dump(_tf, _f, indent=2, ensure_ascii=False)
print(f"../sync/transfer.json: simulation geschrieben")
print(f"  n_years={n_years:.2f} | spread_mean={_tf['simulation'].get('spread_mean_eur_mwh','?')} EUR/MWh")
for seg, v in _tf['simulation']['wirtschaftlichkeit'].items():
    print(f"  {seg}: ROI={v['roi_pct']}% | net={v['net_annual']:.0f} EUR/J")


**Abschlusskontrolle:** Pflicht-Ausgabedateien validieren; dataindex-Status ausgeben.
Fehler müssen vor NB03 behoben werden.


---
## Abschluss & Verifikation<a id='abschluss-verifikation_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Pflicht-Ausgabedateien validieren; Übergabe-Kennzahlen in `../sync/transfer.json` prüfen.

In [ ]:
# ── Abschlusskontrolle NB03 ──────────────────────────────────────────────────
print('NB03 – Abschlusskontrolle')
print('=' * 60)
CLEAN_FILE = os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv')
ECON_FILE  = os.path.join(DIR_INTER_SZ,  'wirtschaftlichkeit.csv')
SZ_FILE    = os.path.join(DIR_INTER_SZ,  'netzentlastung_szenarien.csv')
SPREAD_F   = os.path.join(DIR_INTER,     'spread_zeitreihe.csv')
# min_bytes: gross für Zeitreihen (>10k Zeilen), klein für Tabellen (4 Zeilen)
for path, desc, min_bytes in [
    (CLEAN_FILE, 'ch_spot_prices_clean.csv',                          50_000),
    (ECON_FILE,  f'wirtschaftlichkeit.csv [{GLEICHZEITIGKEIT}]',         100),
    (SZ_FILE,    f'netzentlastung_szenarien.csv [{GLEICHZEITIGKEIT}]',   100),
    (SPREAD_F,   'spread_zeitreihe.csv',                                 500),
]:
    ok = os.path.exists(path) and os.path.getsize(path) >= min_bytes
    print(f'  {"✅" if ok else "❌"}  {desc}')
print()
print('Kür-Abhängigkeiten:')
print('  ch_import_export_analyse.csv → wird in K_02 berechnet (benötigt ch_crossborder_raw.csv)')
print(f'\n→ Weiter mit: NB03 Visualisierungen')


| [← NB02 Daten Bereinigung](02_Daten_Bereinigung.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [NB04 Visualisierungen →](04_Visualisierungen.ipynb) |
|:---|:---:|---:|